In [1]:

# STAR SCHEMA - Rebuild DimDate and FactTaxiTrips from Gold
# Includes precomputed trip duration and distance/duration bins.

from pyspark.sql import functions as F

# ============================================================
# CONFIGURATION
# ============================================================

GOLD_TABLE = "yellow_taxi_gold"
DATE_TABLE = "DimDate"
FACT_TABLE = "FactTaxiTrips"
TEMP_FACT_TABLE = "FactTaxiTrips_Rebuild"
STATS_TABLE = "TaxiKeyStatistics"

# ============================================================
# 1. READ GOLD AND EXISTING DATE DIMENSION
# ============================================================

gold_df = spark.table(GOLD_TABLE)
stats_df = spark.table(STATS_TABLE)
existing_dim_date = spark.table(DATE_TABLE)

gold_rows = gold_df.count()

print(f"Gold source rows: {gold_rows:,}")
print(f"TaxiKeyStatistics rows: {stats_df.count()}")


required_gold_columns = [
    "TripDurationSeconds",
    "TripDistanceBin",
    "TripDistanceBinSort",
    "TripDurationBin",
    "TripDurationBinSort",
    "request_source"
]

required_stats_columns = [
    "TotalTrips",
    "MedianTripDurationSeconds",
    "MedianTripDurationMinutes",
    "MaxTripDurationSeconds",
    "MaxTripDurationMinutes",
    "ZeroDurationTrips",
    "ZeroDurationTripsPct",
    "ZeroDistanceTrips",
    "ZeroDistanceTripsPct",
    "MedianTripDistance",
    "CalculationDate"
]

stats_total_trips = stats_df.first()["TotalTrips"]

if stats_total_trips != gold_rows:
    raise RuntimeError(
        f"TaxiKeyStatistics is stale: "
        f"{stats_total_trips:,} trips in statistics, "
        f"{gold_rows:,} trips in Gold."
    )

print("TaxiKeyStatistics row count matches Gold.")

missing_stats_columns = [
    column
    for column in required_stats_columns
    if column not in stats_df.columns
]

if missing_stats_columns:
    raise RuntimeError(
        "TaxiKeyStatistics is missing columns: "
        + ", ".join(missing_stats_columns)
    )

if stats_df.count() != 1:
    raise RuntimeError(
        "TaxiKeyStatistics must contain exactly one summary row."
    )

print("TaxiKeyStatistics validated successfully.")

missing_columns = [
    column
    for column in required_gold_columns
    if column not in gold_df.columns
]

if missing_columns:
    raise RuntimeError(
        "Gold is missing required columns: "
        + ", ".join(missing_columns)
        + ". Run and validate the Gold notebook first."
    )



# ============================================================
# 2. BUILD A COMPLETE DATE RANGE
# Preserve existing calendar coverage and extend when needed.
# Include both pickup and dropoff dates.
# ============================================================

all_dates = (
    gold_df.select(
        F.to_date("tpep_pickup_datetime").alias("CalendarDate")
    )
    .union(
        gold_df.select(
            F.to_date("tpep_dropoff_datetime").alias("CalendarDate")
        )
    )
    .union(
        existing_dim_date.select(
            F.col("Date").alias("CalendarDate")
        )
    )
)

date_bounds = all_dates.agg(
    F.min("CalendarDate").alias("min_date"),
    F.max("CalendarDate").alias("max_date")
).first()

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

if min_date is None or max_date is None:
    raise RuntimeError("Cannot determine the DimDate range.")

print(f"Calendar range: {min_date} to {max_date}")

dim_date_new = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(min_date),
                F.lit(max_date),
                F.expr("interval 1 day")
            )
        ).alias("Date")
    )
    .withColumn(
        "DateKey",
        F.date_format("Date", "yyyyMMdd").cast("long")
    )
    .withColumn("Year", F.year("Date").cast("long"))
    .withColumn(
        "Quarter",
        F.concat(
            F.lit("Q"),
            F.quarter("Date").cast("string")
        )
    )
    .withColumn("MonthNumber", F.month("Date").cast("long"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Day", F.dayofmonth("Date").cast("long"))
    .withColumn("DayOfWeek", F.date_format("Date", "EEEE"))
    .withColumn(
        "DayOfWeekNumber",
        ((F.dayofweek("Date") + 5) % 7 + 1).cast("long")
    )
    .withColumn(
        "IsWeekend",
        F.dayofweek("Date").isin([1, 7])
    )
    .select(
        "Date",
        "DateKey",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "Day",
        "DayOfWeek",
        "DayOfWeekNumber",
        "IsWeekend"
    )
)

# Validate calendar before writing.
calendar_stats = dim_date_new.agg(
    F.count("*").alias("rows"),
    F.countDistinct("DateKey").alias("distinct_keys"),
    F.min("Date").alias("min_date"),
    F.max("Date").alias("max_date")
).first()

if calendar_stats["rows"] != calendar_stats["distinct_keys"]:
    raise RuntimeError("DimDate contains duplicate DateKey values.")

print(
    f"DimDate validated: {calendar_stats['rows']:,} days, "
    f"{calendar_stats['min_date']} to {calendar_stats['max_date']}"
)

# ============================================================
# 3. BUILD FACT FROM GOLD
# Include all precomputed analytical columns.
# ============================================================

fact_rebuild_df = (
    gold_df
    .withColumn(
        "PickupDateKey",
        F.date_format(
            "tpep_pickup_datetime", "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "DropoffDateKey",
        F.date_format(
            "tpep_dropoff_datetime", "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "VendorID",
        "PickupDateKey",
        "DropoffDateKey",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "TripDurationSeconds",
        "PULocationID",
        "DOLocationID",
        "payment_type",
        "passenger_count",
        "trip_distance",
        "RatecodeID",
        "store_and_fwd_flag",
        "request_source",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "Airport_fee",
        "cbd_congestion_fee",
        "TripDistanceBin",
        "TripDistanceBinSort",
        "TripDurationBin",
        "TripDurationBinSort"
    )
)

# ============================================================
# 4. WRITE TEMPORARY FACT
# Production tables remain untouched until validation passes.
# ============================================================

(
    fact_rebuild_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TEMP_FACT_TABLE)
)

temp_fact = spark.table(TEMP_FACT_TABLE)

# ============================================================
# 5. VALIDATE TEMP FACT
# ============================================================

fact_stats = temp_fact.agg(
    F.count("*").alias("rows"),
    F.sum(
        F.when(F.col("PickupDateKey").isNull(), 1).otherwise(0)
    ).alias("null_pickup_keys"),
    F.sum(
        F.when(F.col("DropoffDateKey").isNull(), 1).otherwise(0)
    ).alias("null_dropoff_keys"),
    F.sum(
        F.when(F.col("TripDurationBin").isNull(), 1).otherwise(0)
    ).alias("null_duration_bins"),
    F.sum(
        F.when(F.col("TripDistanceBin").isNull(), 1).otherwise(0)
    ).alias("null_distance_bins")
).first()

date_keys = dim_date_new.select(
    F.col("DateKey").cast("int").alias("DateKey")
)

pickup_missing = (
    temp_fact.select("PickupDateKey")
    .distinct()
    .join(
        date_keys,
        F.col("PickupDateKey") == date_keys["DateKey"],
        "left_anti"
    )
    .count()
)

dropoff_missing = (
    temp_fact.select("DropoffDateKey")
    .distinct()
    .join(
        date_keys,
        F.col("DropoffDateKey") == date_keys["DateKey"],
        "left_anti"
    )
    .count()
)

print(f"Temporary fact rows: {fact_stats['rows']:,}")
print(f"Null pickup keys: {fact_stats['null_pickup_keys']}")
print(f"Null dropoff keys: {fact_stats['null_dropoff_keys']}")
print(f"Null duration bins: {fact_stats['null_duration_bins']}")
print(f"Null distance bins: {fact_stats['null_distance_bins']}")
print(f"Pickup dates missing from DimDate: {pickup_missing}")
print(f"Dropoff dates missing from DimDate: {dropoff_missing}")

if fact_stats["rows"] != gold_rows:
    raise RuntimeError("Fact row count does not match Gold.")

if fact_stats["null_pickup_keys"] > 0:
    raise RuntimeError("Null PickupDateKey values detected.")

if fact_stats["null_dropoff_keys"] > 0:
    raise RuntimeError("Null DropoffDateKey values detected.")

if fact_stats["null_duration_bins"] > 0:
    raise RuntimeError("Null TripDurationBin values detected.")

if fact_stats["null_distance_bins"] > 0:
    raise RuntimeError("Null TripDistanceBin values detected.")

if pickup_missing > 0 or dropoff_missing > 0:
    raise RuntimeError("Fact contains dates missing from DimDate.")

# ============================================================
# 6. WRITE DIMDATE AND REPLACE PRODUCTION FACT
# Only reached after temporary fact validation succeeds.
# ============================================================

(
    dim_date_new.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DATE_TABLE)
)

(
    temp_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_TABLE)
)

# ============================================================
# 7. FINAL VALIDATION
# ============================================================

final_fact = spark.table(FACT_TABLE)

final_rows = final_fact.count()
final_date_rows = spark.table(DATE_TABLE).count()

expected_fact_columns = [
    "TripDurationSeconds",
    "TripDistanceBin",
    "TripDistanceBinSort",
    "TripDurationBin",
    "TripDurationBinSort",
    "request_source"
]

missing_final_columns = [
    column
    for column in expected_fact_columns
    if column not in final_fact.columns
]

print(f"Final FactTaxiTrips rows: {final_rows:,}")
print(f"Final DimDate rows: {final_date_rows:,}")
print("Final analytical columns:", expected_fact_columns)

if final_rows != gold_rows:
    raise RuntimeError("Final fact row count does not match Gold.")

if missing_final_columns:
    raise RuntimeError(
        "Final FactTaxiTrips is missing columns: "
        + ", ".join(missing_final_columns)
    )

print("SUCCESS: DimDate and FactTaxiTrips refreshed.")
print("SUCCESS: Duration and distance bins included in the star schema.")


StatementMeta(, 98d85f8f-59bf-4fe7-aab2-c0f83148b311, 3, Finished, Available, Finished, False)

Gold source rows: 78,423,712
TaxiKeyStatistics rows: 1
TaxiKeyStatistics row count matches Gold.
TaxiKeyStatistics validated successfully.
Calendar range: 2001-01-01 to 2026-09-01
DimDate validated: 9,375 days, 2001-01-01 to 2026-09-01
Temporary fact rows: 78,423,712
Null pickup keys: 0
Null dropoff keys: 0
Null duration bins: 0
Null distance bins: 0
Pickup dates missing from DimDate: 0
Dropoff dates missing from DimDate: 0
Final FactTaxiTrips rows: 78,423,712
Final DimDate rows: 9,375
Final analytical columns: ['TripDurationSeconds', 'TripDistanceBin', 'TripDistanceBinSort', 'TripDurationBin', 'TripDurationBinSort', 'request_source']
SUCCESS: DimDate and FactTaxiTrips refreshed.
SUCCESS: Duration and distance bins included in the star schema.


In [ ]:
fact_df = spark.table("FactTaxiTrips")

print("TripDurationSeconds" in fact_df.columns)

fact_df.select(
    "TripDurationSeconds"
).show(10)

StatementMeta(, 0e8d7573-6a7b-41bd-b967-87813d3f0d04, 4, Finished, Available, Finished, False)

True
+-------------------+
|TripDurationSeconds|
+-------------------+
|                352|
|               2262|
|                804|
|               1018|
|                951|
|               1664|
|               1858|
|                779|
|                672|
|                691|
+-------------------+
only showing top 10 rows
